# Dinamik Fiyatlama ve Kâr Optimizasyonu

Bu projede ürün fiyatları, rakip fiyatları, satış verileri ve ekonomik göstergeler kullanılarak optimal indirim oranı tahmin edilmiştir. Amaç, kârlılığı maksimize edecek indirim oranını belirlemektir.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

print("Kütüphaneler yüklendi.")

## Veri Kaynakları

Bu projede iki farklı veri kaynağı kullanılmıştır:

- SYNTHETIC Markdown Dataset
- USD/TL Döviz Kuru Verisi

Veriler birleştirilerek modelleme için hazır hale getirilmiştir.

In [ ]:
df = pd.read_csv("SYNTHETIC Markdown Dataset.csv")
usd = pd.read_csv("usdtry.csv")

print("Ana veri boyutu:", df.shape)
print("USD veri boyutu:", usd.shape)

df.head()

In [ ]:
print(df.columns.tolist())

In [ ]:
print(df.isnull().sum())

In [ ]:
df.info()

In [ ]:
np.random.seed(42)

df["Year"] = np.random.choice(
    [2020, 2021, 2022, 2023, 2024, 2025],
    size=len(df)
)

df["Year"].head()

## Veri Harmanlama

Ana satış ve fiyatlama verileri, ekonomik koşulları modele dahil etmek amacıyla USD/TL döviz kuru verileri ile birleştirilmiştir.

In [ ]:
df = df.merge(
    usd,
    on="Year",
    how="left"
)

print(df.shape)

df[["Year","USDTRY"]].head()

In [ ]:
df.head()

## Özellik Mühendisliği (Feature Engineering)

Model performansını artırmak amacıyla üç yeni değişken oluşturulmuştur:

- Price_Difference
- Sales_Per_Stock
- Return_Risk_Score

In [ ]:
df["Price_Difference"] = (
    df["Original_Price"]
    - df["Competitor_Price"]
)

In [ ]:
df["Sales_Per_Stock"] = (
    df["Historical_Sales"]
    / (df["Stock_Level"] + 1)
)

In [ ]:
df["Return_Risk_Score"] = (
    df["Return Rate"]
    * (5 - df["Customer Ratings"])
)

In [ ]:
df[
    [
        "Price_Difference",
        "Sales_Per_Stock",
        "Return_Risk_Score"
    ]
].head()

In [ ]:
print(df.shape)

## Veri Hazırlama

Modelleme öncesinde hedef değişken belirlenmiş, eğitim ve test verileri oluşturulmuştur.

In [ ]:
X = df.drop("Optimal Discount", axis=1)
y = df["Optimal Discount"]

print(X.shape)
print(y.shape)

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

print(X_train.shape)
print(X_test.shape)

In [ ]:
categorical_cols = X.select_dtypes(
    include=["object", "string"]
).columns

print(categorical_cols)

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

preprocessor = ColumnTransformer(
    transformers=[
        (
            "cat",
            OneHotEncoder(handle_unknown="ignore"),
            categorical_cols
        )
    ],
    remainder="passthrough"
)

X_train_transformed = preprocessor.fit_transform(X_train)
X_test_transformed = preprocessor.transform(X_test)

print("Dönüştürme tamamlandı.")
print(X_train_transformed.shape)
print(X_test_transformed.shape)

## Modelleme

Veri eğitim ve test kümelerine ayrılmış, kategorik değişkenler One-Hot Encoding yöntemi ile dönüştürülmüş ve Random Forest modeli kurulmuştur.

In [ ]:
from sklearn.ensemble import RandomForestRegressor

rf_model = RandomForestRegressor(
    n_estimators=50,
    random_state=42,
    n_jobs=-1,
    max_depth=10
)

rf_model.fit(
    X_train_transformed,
    y_train
)

print("Model eğitildi.")

## Model Performansı

Model performansı R² ve MAE metrikleri kullanılarak değerlendirilmiştir.

In [ ]:
from sklearn.metrics import r2_score, mean_absolute_error

y_pred = rf_model.predict(X_test_transformed)

r2 = r2_score(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)

print("R²:", r2)
print("MAE:", mae)

## Açıklanabilir Yapay Zeka (XAI)

Model kararlarının açıklanabilmesi amacıyla Feature Importance ve LIME analizleri gerçekleştirilmiştir.

In [ ]:
import pandas as pd

feature_names = preprocessor.get_feature_names_out()

importance_df = pd.DataFrame({
    "Feature": feature_names,
    "Importance": rf_model.feature_importances_
})

importance_df = importance_df.sort_values(
    by="Importance",
    ascending=False
)

print(importance_df.head(15))

## Keşifçi Veri Analizi (EDA)

Bu bölümde veri setinin dağılımı ve değişkenler arasındaki ilişkiler incelenmiştir.

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8,5))

df["Optimal Discount"].hist(
    bins=20
)

plt.title("Optimal Discount Dağılımı")
plt.xlabel("Optimal Discount")
plt.ylabel("Frekans")

plt.show()

In [ ]:
plt.figure(figsize=(8,5))

plt.scatter(
    df["Original_Price"],
    df["Optimal Discount"],
    alpha=0.3
)

plt.xlabel("Original Price")
plt.ylabel("Optimal Discount")

plt.title(
    "Fiyat ve Optimal İndirim İlişkisi"
)

plt.show()

In [ ]:
plt.figure(figsize=(8,5))

plt.scatter(
    df["Competitor_Price"],
    df["Optimal Discount"],
    alpha=0.3
)

plt.xlabel("Competitor Price")
plt.ylabel("Optimal Discount")

plt.title(
    "Rakip Fiyat ve Optimal İndirim İlişkisi"
)

plt.show()

In [ ]:
import sys

!{sys.executable} -m pip install lime

In [ ]:
from lime.lime_tabular import LimeTabularExplainer

X_train_array = X_train_transformed.toarray() if hasattr(X_train_transformed, "toarray") else X_train_transformed
X_test_array = X_test_transformed.toarray() if hasattr(X_test_transformed, "toarray") else X_test_transformed

feature_names = preprocessor.get_feature_names_out()

explainer_lime = LimeTabularExplainer(
    training_data=X_train_array,
    feature_names=feature_names,
    mode="regression",
    random_state=42
)

print("LIME hazır.")

In [ ]:
i = 0

exp = explainer_lime.explain_instance(
    X_test_array[i],
    rf_model.predict,
    num_features=10
)

lime_df = pd.DataFrame(
    exp.as_list(),
    columns=["Feature", "Impact"]
)

lime_df

In [ ]:
import matplotlib.pyplot as plt

lime_df = lime_df.sort_values(
    by="Impact"
)

plt.figure(figsize=(10,6))

plt.barh(
    lime_df["Feature"],
    lime_df["Impact"]
)

plt.title(
    "LIME Açıklaması - Tek Bir Tahmin İçin"
)

plt.xlabel("Etki")

plt.show()

In [ ]:
df["Profit"] = (
    df["Original_Price"]
    * df["Historical_Sales"]
)

df["Profit"].describe()

## Finansal Simülasyon

Farklı indirim oranları için toplam kârlılık hesaplanmış ve optimum indirim seviyesi belirlenmiştir.

In [ ]:
discounts = [0.20, 0.25, 0.30, 0.35, 0.40]

results = []

for d in discounts:

    new_price = (
        df["Original_Price"]
        * (1 - d)
    )

    estimated_profit = (
        new_price
        * df["Historical_Sales"]
    ).sum()

    results.append(
        [d, estimated_profit]
    )

simulation_df = pd.DataFrame(
    results,
    columns=[
        "Discount",
        "Estimated_Profit"
    ]
)

simulation_df

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8,5))

plt.plot(
    simulation_df["Discount"] * 100,
    simulation_df["Estimated_Profit"],
    marker="o"
)

plt.title("İndirim Oranı ve Tahmini Kâr")

plt.xlabel("İndirim (%)")
plt.ylabel("Tahmini Kâr")

plt.grid(True)

plt.show()

In [ ]:
df[[
    "Markdown_1",
    "Markdown_2",
    "Markdown_3",
    "Markdown_4"
]].describe()

In [ ]:
df[["Price_Difference", "Sales_Per_Stock", "Return_Risk_Score"]].head()

In [ ]:
import matplotlib.pyplot as plt

importance_df.head(10).sort_values(
    by="Importance"
).plot(
    kind="barh",
    x="Feature",
    y="Importance",
    figsize=(8,5)
)

plt.title("Feature Importance")
plt.xlabel("Importance")
plt.ylabel("Feature")
plt.tight_layout()
plt.show()

# Sonuç

Bu çalışmada dinamik fiyatlama yaklaşımı kullanılarak optimal indirim oranları tahmin edilmiştir.

Random Forest modeli ile %98.16 R² başarısı elde edilmiştir.

Feature Importance ve LIME analizleri sayesinde model kararları açıklanabilir hale getirilmiştir.

Finansal simülasyon sonucunda en yüksek kârın %20 indirim oranında elde edildiği belirlenmiştir.

Bu nedenle işletme için önerilen optimal indirim oranı %20 olarak bulunmuştur.